In [ ]:
# ============================================================
# 0. INSTALLATION
# ============================================================
!pip install -q ucimlrepo scikit-learn joblib



In [ ]:
# ============================================================
# 1. IMPORTS AND OUTPUT DIRECTORIES
# ============================================================

import os
import json
import warnings
import shutil
import joblib
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
CV_FOLDS = 5

BASE_DIR = "/content/data_scientist_outputs"
CSV_DIR = os.path.join(BASE_DIR, "csv")
MODEL_DIR = "/content/models"

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Data Scientist environment ready.")



Data Scientist environment ready.


In [ ]:
# ============================================================
# 2. DATA ACQUISITION
# ============================================================

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features.copy()
y_raw = heart_disease.data.targets.copy()

print("Dataset shape:", X.shape)
print("Target shape:", y_raw.shape)

display(X.head())
display(y_raw.head())



Dataset shape: (303, 13)
Target shape: (303, 1)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


,num
0,0
1,2
2,1
3,0
4,0


In [ ]:
# ============================================================
# 3. TARGET TRANSFORMATION
# ============================================================

# Original UCI target:
# 0 = no disease
# 1-4 = disease severity
#
# For this binary classification task:
# num == 0 -> 0 (No Disease)
# num > 0  -> 1 (Disease)

y = (y_raw["num"] > 0).astype(int)

target_labels = {
    0: "No Disease",
    1: "Disease"
}

target_counts = y.map(target_labels).value_counts()

target_distribution = pd.DataFrame({
    "Target": target_counts.index,
    "Count": target_counts.values,
    "Percentage": (
        target_counts.values / len(y) * 100
    ).round(2)
})

display(target_distribution)

target_distribution.to_csv(
    os.path.join(CSV_DIR, "target_distribution.csv"),
    index=False
)



,Target,Count,Percentage
0,No Disease,164,54.13
1,Disease,139,45.87


In [ ]:
# ============================================================
# 4. FEATURE GROUPS
# ============================================================

numeric_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)



Numeric features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


In [ ]:
# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

split_report = pd.DataFrame([
    {
        "Split": "Train",
        "Samples": len(y_train),
        "No Disease %": round((y_train == 0).mean() * 100, 2),
        "Disease %": round((y_train == 1).mean() * 100, 2)
    },
    {
        "Split": "Test",
        "Samples": len(y_test),
        "No Disease %": round((y_test == 0).mean() * 100, 2),
        "Disease %": round((y_test == 1).mean() * 100, 2)
    }
])

display(split_report)

split_report.to_csv(
    os.path.join(CSV_DIR, "train_test_split_summary.csv"),
    index=False
)



Training samples: 242
Testing samples: 61


,Split,Samples,No Disease %,Disease %
0,Train,242,54.13,45.87
1,Test,61,54.10,45.90


In [ ]:
# ============================================================
# 6. LEAKAGE-SAFE PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")
print("Imputation and scaling/encoding occur inside each CV fold.")



Preprocessing pipeline created.
Imputation and scaling/encoding occur inside each CV fold.


In [ ]:
# ============================================================
# 7. MODELING STRATEGY AND ALGORITHM JUSTIFICATION
# ============================================================

algorithm_justification = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Role": "Baseline",
        "Why included": "Simple, interpretable linear classifier and reference point.",
        "Bias_Variance": "Lower variance and potentially higher bias when relationships are nonlinear."
    },
    {
        "Model": "Decision Tree",
        "Role": "Candidate",
        "Why included": "Captures nonlinear relationships and interactions.",
        "Bias_Variance": "Can have low bias but high variance when the tree is complex."
    },
    {
        "Model": "Random Forest",
        "Role": "Candidate",
        "Why included": "Ensemble of trees designed to improve robustness and model nonlinear patterns.",
        "Bias_Variance": "Usually reduces variance relative to a single decision tree."
    },
    {
        "Model": "SVM",
        "Role": "Candidate",
        "Why included": "Can model complex decision boundaries using kernels.",
        "Bias_Variance": "Regularization parameter C controls the bias-variance tradeoff."
    }
])

display(algorithm_justification)

algorithm_justification.to_csv(
    os.path.join(CSV_DIR, "algorithm_justification.csv"),
    index=False
)



,Model,Role,Why included,Bias_Variance
0,Logistic Regression,Baseline,"Simple, interpretable linear classifier and re...",Lower variance and potentially higher bias whe...
1,Decision Tree,Candidate,Captures nonlinear relationships and interacti...,Can have low bias but high variance when the t...
2,Random Forest,Candidate,Ensemble of trees designed to improve robustne...,Usually reduces variance relative to a single ...
3,SVM,Candidate,Can model complex decision boundaries using ke...,Regularization parameter C controls the bias-v...


In [ ]:
# ============================================================
# 8. CROSS-VALIDATION SETUP
# ============================================================

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print(f"{CV_FOLDS}-fold Stratified Cross-Validation configured.")



5-fold Stratified Cross-Validation configured.


In [ ]:
# ============================================================
# 9. BASELINE MODEL: LOGISTIC REGRESSION
# ============================================================

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

baseline_cv = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=True
)

baseline_result = {
    "Model": "Logistic Regression - Baseline",
    "CV Accuracy Mean": baseline_cv["test_accuracy"].mean(),
    "CV Accuracy Std": baseline_cv["test_accuracy"].std(),
    "CV Precision": baseline_cv["test_precision"].mean(),
    "CV Recall": baseline_cv["test_recall"].mean(),
    "CV F1": baseline_cv["test_f1"].mean(),
    "CV F1 Std": baseline_cv["test_f1"].std(),
    "CV ROC-AUC": baseline_cv["test_roc_auc"].mean(),
    "Train Accuracy": baseline_cv["train_accuracy"].mean(),
    "Train F1": baseline_cv["train_f1"].mean()
}

baseline_result["Train-CV Accuracy Gap"] = (
    baseline_result["Train Accuracy"] -
    baseline_result["CV Accuracy Mean"]
)

baseline_result["Train-CV F1 Gap"] = (
    baseline_result["Train F1"] -
    baseline_result["CV F1"]
)

baseline_results_df = pd.DataFrame([baseline_result])

display(baseline_results_df.round(4))

baseline_results_df.to_csv(
    os.path.join(CSV_DIR, "baseline_cv_results.csv"),
    index=False
)



,Model,CV Accuracy Mean,CV Accuracy Std,CV Precision,CV Recall,CV F1,CV F1 Std,CV ROC-AUC,Train Accuracy,Train F1,Train-CV Accuracy Gap,Train-CV F1 Gap
0,Logistic Regression - Baseline,0.8471,0.01,0.8768,0.7834,0.8245,0.0069,0.9025,0.8688,0.8526,0.0217,0.0281


In [ ]:
# ============================================================
# 10. UNTUNED CANDIDATE MODELS
# ============================================================

candidate_models = {
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "SVM": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", SVC(
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])
}

untuned_results = []

for model_name, model in candidate_models.items():

    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "Model": model_name,
        "CV Accuracy Mean": results["test_accuracy"].mean(),
        "CV Accuracy Std": results["test_accuracy"].std(),
        "CV Precision": results["test_precision"].mean(),
        "CV Recall": results["test_recall"].mean(),
        "CV F1": results["test_f1"].mean(),
        "CV F1 Std": results["test_f1"].std(),
        "CV ROC-AUC": results["test_roc_auc"].mean(),
        "Train Accuracy": results["train_accuracy"].mean(),
        "Train F1": results["train_f1"].mean()
    }

    row["Train-CV Accuracy Gap"] = (
        row["Train Accuracy"] - row["CV Accuracy Mean"]
    )

    row["Train-CV F1 Gap"] = (
        row["Train F1"] - row["CV F1"]
    )

    untuned_results.append(row)

untuned_comparison = pd.DataFrame(untuned_results)

display(
    untuned_comparison.sort_values(
        "CV F1", ascending=False
    ).round(4)
)



,Model,CV Accuracy Mean,CV Accuracy Std,CV Precision,CV Recall,CV F1,CV F1 Std,CV ROC-AUC,Train Accuracy,Train F1,Train-CV Accuracy Gap,Train-CV F1 Gap
1,Random Forest,0.8181,0.0307,0.8173,0.7826,0.7948,0.0469,0.8972,1.0000,1.0000,0.1819,0.2052
2,SVM,0.8139,0.0235,0.8303,0.7648,0.7893,0.0271,0.8884,0.9122,0.9003,0.0982,0.1109
0,Decision Tree,0.6980,0.0596,0.6782,0.6842,0.6768,0.0420,0.6967,1.0000,1.0000,0.3020,0.3232


In [ ]:
# ============================================================
# 11. BASELINE VS UNTUNED CANDIDATES
# ============================================================

untuned_comparison_all = pd.concat(
    [
        baseline_results_df,
        untuned_comparison
    ],
    ignore_index=True
)

untuned_comparison_all = untuned_comparison_all.sort_values(
    "CV F1",
    ascending=False
)

display(untuned_comparison_all.round(4))

untuned_comparison_all.to_csv(
    os.path.join(
        CSV_DIR,
        "cross_validation_comparison.csv"
    ),
    index=False
)



,Model,CV Accuracy Mean,CV Accuracy Std,CV Precision,CV Recall,CV F1,CV F1 Std,CV ROC-AUC,Train Accuracy,Train F1,Train-CV Accuracy Gap,Train-CV F1 Gap
0,Logistic Regression - Baseline,0.8471,0.0100,0.8768,0.7834,0.8245,0.0069,0.9025,0.8688,0.8526,0.0217,0.0281
2,Random Forest,0.8181,0.0307,0.8173,0.7826,0.7948,0.0469,0.8972,1.0000,1.0000,0.1819,0.2052
3,SVM,0.8139,0.0235,0.8303,0.7648,0.7893,0.0271,0.8884,0.9122,0.9003,0.0982,0.1109
1,Decision Tree,0.6980,0.0596,0.6782,0.6842,0.6768,0.0420,0.6967,1.0000,1.0000,0.3020,0.3232


## for a second performance estimate.


In [ ]:
# ============================================================

dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

dt_param_grid = {
    "classifier__max_depth": [2, 3, 4, 5, 6, 8, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4, 8],
    "classifier__criterion": ["gini", "entropy"]
}

dt_grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

dt_grid.fit(X_train, y_train)

print("Decision Tree best parameters:")
print(dt_grid.best_params_)
print("Best CV F1:", round(dt_grid.best_score_, 4))

pd.DataFrame(dt_grid.cv_results_).to_csv(
    os.path.join(
        CSV_DIR,
        "decision_tree_tuning_results.csv"
    ),
    index=False
)


rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10, 15],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

rf_grid.fit(X_train, y_train)

print("\nRandom Forest best parameters:")
print(rf_grid.best_params_)
print("Best CV F1:", round(rf_grid.best_score_, 4))

pd.DataFrame(rf_grid.cv_results_).to_csv(
    os.path.join(
        CSV_DIR,
        "random_forest_tuning_results.csv"
    ),
    index=False
)


svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        probability=True,
        random_state=RANDOM_STATE
    ))
])

svm_param_grid = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 0.001, 0.01, 0.1],
    "classifier__kernel": ["rbf", "linear"]
}

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

svm_grid.fit(X_train, y_train)

print("\nSVM best parameters:")
print(svm_grid.best_params_)
print("Best CV F1:", round(svm_grid.best_score_, 4))

pd.DataFrame(svm_grid.cv_results_).to_csv(
    os.path.join(
        CSV_DIR,
        "svm_tuning_results.csv"
    ),
    index=False
)



Decision Tree best parameters:
{'classifier__criterion': 'entropy', 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 8, 'classifier__min_samples_split': 2}
Best CV F1: 0.7752

Random Forest best parameters:
{'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 300}
Best CV F1: 0.8035

SVM best parameters:
{'classifier__C': 10, 'classifier__gamma': 0.01, 'classifier__kernel': 'rbf'}
Best CV F1: 0.827


In [ ]:
# ============================================================
# 13. TUNED MODEL COMPARISON
# ============================================================

tuned_grids = {
    "Decision Tree": dt_grid,
    "Random Forest": rf_grid,
    "SVM": svm_grid
}

tuned_results = []

for model_name, grid in tuned_grids.items():

    i = grid.best_index_

    tuned_results.append({
        "Model": model_name,
        "CV Accuracy Mean": grid.cv_results_["mean_test_accuracy"][i],
        "CV Accuracy Std": grid.cv_results_["std_test_accuracy"][i],
        "CV Precision": grid.cv_results_["mean_test_precision"][i],
        "CV Recall": grid.cv_results_["mean_test_recall"][i],
        "CV F1": grid.cv_results_["mean_test_f1"][i],
        "CV F1 Std": grid.cv_results_["std_test_f1"][i],
        "CV ROC-AUC": grid.cv_results_["mean_test_roc_auc"][i],
        "Train Accuracy": grid.cv_results_["mean_train_accuracy"][i],
        "Train F1": grid.cv_results_["mean_train_f1"][i],
        "Train-CV Accuracy Gap": (
            grid.cv_results_["mean_train_accuracy"][i]
            - grid.cv_results_["mean_test_accuracy"][i]
        ),
        "Train-CV F1 Gap": (
            grid.cv_results_["mean_train_f1"][i]
            - grid.cv_results_["mean_test_f1"][i]
        ),
        "Best Parameters": str(grid.best_params_)
    })

tuned_comparison = pd.DataFrame(tuned_results)

display(
    tuned_comparison.sort_values(
        "CV F1", ascending=False
    ).round(4)
)

tuned_comparison.to_csv(
    os.path.join(
        CSV_DIR,
        "tuned_model_comparison.csv"
    ),
    index=False
)



,Model,CV Accuracy Mean,CV Accuracy Std,CV Precision,CV Recall,CV F1,CV F1 Std,CV ROC-AUC,Train Accuracy,Train F1,Train-CV Accuracy Gap,Train-CV F1 Gap,Best Parameters
2,SVM,0.8511,0.0247,0.8848,0.7830,0.8270,0.0330,0.8968,0.8730,0.8551,0.0218,0.0280,"{'classifier__C': 10, 'classifier__gamma': 0.0..."
1,Random Forest,0.8264,0.0172,0.8318,0.7830,0.8035,0.0305,0.8946,1.0000,1.0000,0.1736,0.1965,"{'classifier__max_depth': None, 'classifier__m..."
0,Decision Tree,0.7977,0.0411,0.7878,0.7735,0.7752,0.0589,0.8594,0.8574,0.8407,0.0597,0.0654,"{'classifier__criterion': 'entropy', 'classifi..."


In [ ]:
# ============================================================
# 14. FINAL MODEL COMPARISON
# ============================================================

baseline_for_final = baseline_results_df.copy()
baseline_for_final["Tuning_Status"] = "Baseline"
baseline_for_final["Best Parameters"] = "Default baseline configuration"

tuned_for_final = tuned_comparison.copy()
tuned_for_final["Tuning_Status"] = "Tuned"

final_comparison = pd.concat(
    [
        baseline_for_final,
        tuned_for_final
    ],
    ignore_index=True
)

final_comparison = final_comparison.sort_values(
    "CV F1",
    ascending=False
)

display(final_comparison.round(4))

final_comparison.to_csv(
    os.path.join(
        CSV_DIR,
        "final_model_comparison.csv"
    ),
    index=False
)



,Model,CV Accuracy Mean,CV Accuracy Std,CV Precision,CV Recall,CV F1,CV F1 Std,CV ROC-AUC,Train Accuracy,Train F1,Train-CV Accuracy Gap,Train-CV F1 Gap,Tuning_Status,Best Parameters
3,SVM,0.8511,0.0247,0.8848,0.7830,0.8270,0.0330,0.8968,0.8730,0.8551,0.0218,0.0280,Tuned,"{'classifier__C': 10, 'classifier__gamma': 0.0..."
0,Logistic Regression - Baseline,0.8471,0.0100,0.8768,0.7834,0.8245,0.0069,0.9025,0.8688,0.8526,0.0217,0.0281,Baseline,Default baseline configuration
2,Random Forest,0.8264,0.0172,0.8318,0.7830,0.8035,0.0305,0.8946,1.0000,1.0000,0.1736,0.1965,Tuned,"{'classifier__max_depth': None, 'classifier__m..."
1,Decision Tree,0.7977,0.0411,0.7878,0.7735,0.7752,0.0589,0.8594,0.8574,0.8407,0.0597,0.0654,Tuned,"{'classifier__criterion': 'entropy', 'classifi..."


## The test set remains untouched until the next section.


In [ ]:
# ============================================================

selection_pool = pd.concat(
    [
        pd.DataFrame([{
            "Model": "Logistic Regression - Baseline",
            "CV F1": baseline_result["CV F1"],
            "Source": "Baseline"
        }]),
        tuned_comparison[
            ["Model", "CV F1"]
        ].assign(Source="Tuned")
    ],
    ignore_index=True
)

best_selection = selection_pool.loc[
    selection_pool["CV F1"].idxmax()
]

selected_model_name = best_selection["Model"]

if selected_model_name == "Logistic Regression - Baseline":
    final_model = baseline_model
    selected_grid = None
    best_cv_f1 = baseline_result["CV F1"]
    best_parameters = {"classifier": "LogisticRegression baseline"}
else:
    selected_grid = tuned_grids[selected_model_name]
    final_model = selected_grid.best_estimator_
    best_cv_f1 = selected_grid.best_score_
    best_parameters = selected_grid.best_params_

print("Selected final model:", selected_model_name)
print("Selection metric: 5-fold CV F1")
print("Best CV F1:", round(best_cv_f1, 4))
print("Best parameters:")
print(best_parameters)

best_hyperparameters = pd.DataFrame([
    {
        "Model": name,
        "Best CV F1": grid.best_score_,
        "Best Parameters": str(grid.best_params_)
    }
    for name, grid in tuned_grids.items()
])

best_hyperparameters.to_csv(
    os.path.join(
        CSV_DIR,
        "best_hyperparameters.csv"
    ),
    index=False
)



Selected final model: SVM
Selection metric: 5-fold CV F1
Best CV F1: 0.827
Best parameters:
{'classifier__C': 10, 'classifier__gamma': 0.01, 'classifier__kernel': 'rbf'}


In [ ]:
# ============================================================
# 16. BIAS-VARIANCE DIAGNOSTIC
# ============================================================

bias_variance = final_comparison[
    [
        "Model",
        "Tuning_Status",
        "Train Accuracy",
        "CV Accuracy Mean",
        "Train-CV Accuracy Gap",
        "Train F1",
        "CV F1",
        "Train-CV F1 Gap"
    ]
].copy()

bias_variance["Potential_High_Variance"] = (
    bias_variance["Train-CV Accuracy Gap"] >= 0.10
)

display(bias_variance.round(4))

bias_variance.to_csv(
    os.path.join(
        CSV_DIR,
        "bias_variance_diagnostic.csv"
    ),
    index=False
)



,Model,Tuning_Status,Train Accuracy,CV Accuracy Mean,Train-CV Accuracy Gap,Train F1,CV F1,Train-CV F1 Gap,Potential_High_Variance
3,SVM,Tuned,0.8730,0.8511,0.0218,0.8551,0.8270,0.0280,False
0,Logistic Regression - Baseline,Baseline,0.8688,0.8471,0.0217,0.8526,0.8245,0.0281,False
2,Random Forest,Tuned,1.0000,0.8264,0.1736,1.0000,0.8035,0.1965,True
1,Decision Tree,Tuned,0.8574,0.7977,0.0597,0.8407,0.7752,0.0654,False


In [ ]:
# ============================================================
# 17A. FIT FINAL SELECTED MODEL ON COMPLETE TRAINING SET
# ============================================================

# GridSearchCV fits its selected estimator when refit=True.
# If the baseline Logistic Regression is selected, however, it
# has not yet been fitted. Fitting here makes the workflow safe
# for either possible final model.

final_model.fit(X_train, y_train)

print("Final selected model fitted on all training data:")
print(selected_model_name)



In [ ]:
# ============================================================
# 17B. MODEL SELECTION SUMMARY
# ============================================================

selection_pool_display = selection_pool.copy()
selection_pool_display["Selected"] = (
    selection_pool_display["Model"] == selected_model_name
)

if len(selection_pool_display) > 1:
    ordered = selection_pool_display.sort_values(
        "CV F1", ascending=False
    ).reset_index(drop=True)
    top_f1 = ordered.loc[0, "CV F1"]
    second_f1 = ordered.loc[1, "CV F1"]
    selection_margin = top_f1 - second_f1
else:
    selection_margin = np.nan

selection_summary = pd.DataFrame([{
    "Selected_Model": selected_model_name,
    "Selection_Metric": "5-fold CV F1",
    "Selected_CV_F1": best_cv_f1,
    "Margin_Over_Second_Best_CV_F1": selection_margin
}])

display(selection_summary.round(4))

selection_summary.to_csv(
    os.path.join(CSV_DIR, "model_selection_summary.csv"),
    index=False
)



In [ ]:
# ============================================================
# 17. FINAL EVALUATION ON UNTOUCHED TEST SET
# ============================================================

# The test set has not been used for tuning or model selection.

y_pred = final_model.predict(X_test)

if hasattr(final_model, "predict_proba"):
    y_probability = final_model.predict_proba(X_test)[:, 1]
else:
    y_probability = None

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, zero_division=0)
test_recall = recall_score(y_test, y_pred, zero_division=0)
test_f1 = f1_score(y_test, y_pred, zero_division=0)

if y_probability is not None:
    test_roc_auc = roc_auc_score(y_test, y_probability)
else:
    test_roc_auc = np.nan

final_test_results = pd.DataFrame([{
    "Model": selected_model_name,
    "Test Samples": len(y_test),
    "Test Accuracy": test_accuracy,
    "Test Precision": test_precision,
    "Test Recall": test_recall,
    "Test F1": test_f1,
    "Test ROC-AUC": test_roc_auc
}])

display(final_test_results.round(4))

final_test_results.to_csv(
    os.path.join(
        CSV_DIR,
        "final_test_metrics.csv"
    ),
    index=False
)



In [ ]:
# ============================================================
# 18. CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    y_test,
    y_pred,
    target_names=["No Disease", "Disease"],
    output_dict=True,
    zero_division=0
)

classification_report_df = (
    pd.DataFrame(report)
    .transpose()
    .reset_index()
    .rename(columns={"index": "Class"})
)

display(classification_report_df.round(4))

classification_report_df.to_csv(
    os.path.join(
        CSV_DIR,
        "classification_report.csv"
    ),
    index=False
)



In [ ]:
# ============================================================
# 19. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Disease", "Actual Disease"],
    columns=["Predicted No Disease", "Predicted Disease"]
)

display(cm_df)

cm_df.to_csv(
    os.path.join(
        CSV_DIR,
        "confusion_matrix.csv"
    )
)



In [23]:
# ============================================================
# 20. TEST PREDICTIONS
# ============================================================

prediction_output = X_test.copy()
prediction_output["actual"] = y_test.values
prediction_output["predicted"] = y_pred

if y_probability is not None:
    prediction_output["prediction_probability"] = y_probability

prediction_output.to_csv(
    os.path.join(
        CSV_DIR,
        "test_predictions.csv"
    ),
    index=False
)

display(prediction_output.head())



,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,actual,predicted,prediction_probability
219,59,1,4,138,271,0,2,182,0,0.0,1,0.0,3.0,0,0,0.247957
271,66,1,4,160,228,0,2,138,0,2.3,1,0.0,6.0,0,1,0.555222
89,51,0,3,130,256,0,2,149,0,0.5,1,0.0,3.0,0,0,0.039773
101,34,1,1,118,182,0,2,174,0,0.0,1,0.0,3.0,0,0,0.058444
67,54,1,3,150,232,0,2,165,0,1.6,1,0.0,7.0,0,0,0.357354


In [24]:
# ============================================================
# 21. MODEL METADATA
# ============================================================

model_metadata = {
    "project": "UCI Heart Disease Classification",
    "group": "Group 6",
    "role": "Data Scientist",
    "dataset": "UCI Heart Disease Dataset",
    "dataset_id": 45,
    "samples": int(len(X)),
    "features": int(X.shape[1]),
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "target_definition": "0 = No Disease, 1 = Disease; original num > 0 mapped to 1",
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "cv_method": "StratifiedKFold",
    "cv_folds": CV_FOLDS,
    "selection_metric": "F1",
    "selected_model": selected_model_name,
    "best_cv_f1": float(best_cv_f1),
    "best_parameters": best_parameters,
    "test_accuracy": float(test_accuracy),
    "test_precision": float(test_precision),
    "test_recall": float(test_recall),
    "test_f1": float(test_f1),
    "test_roc_auc": float(test_roc_auc) if not np.isnan(test_roc_auc) else None,
    "preprocessing": {
        "numeric_imputation": "median",
        "numeric_scaling": "StandardScaler",
        "categorical_imputation": "most_frequent",
        "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')"
    }
}

metadata_path = os.path.join(
    MODEL_DIR,
    "model_metadata_v1.json"
)

with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=4, default=str)

print("Metadata saved:", metadata_path)



Metadata saved: /content/models/model_metadata_v1.json


In [25]:
# ============================================================
# 22. SERIALIZE FINAL MODEL
# ============================================================

model_path = os.path.join(
    MODEL_DIR,
    "heart_disease_final_model_v1.pkl"
)

joblib.dump(final_model, model_path)

print("Final model saved:", model_path)



Final model saved: /content/models/heart_disease_final_model_v1.pkl


In [26]:
# ============================================================
# 23. MODELING STRATEGY REPORT
# ============================================================

strategy_text = f'''
# Data Scientist Modeling Strategy

## Dataset
UCI Heart Disease Dataset, 303 observations and 13 input features.

## Target
The original `num` target is converted to binary classification:
- 0 = No Disease
- 1 = Disease

## Data Split
An 80/20 stratified train/test split is used.

## Preprocessing
Numeric features:
- median imputation
- StandardScaler

Categorical features:
- most-frequent imputation
- OneHotEncoder with unknown-category handling

Preprocessing is placed inside sklearn Pipelines so it is fitted
within each training fold rather than before cross-validation.

## Models
- Logistic Regression: baseline
- Decision Tree: nonlinear candidate
- Random Forest: ensemble candidate
- SVM: kernel-based candidate

## Cross-Validation
5-fold StratifiedKFold with shuffle=True and random_state=42.

The primary selection metric is F1.

## Hyperparameter Tuning
GridSearchCV is applied to Decision Tree, Random Forest and SVM.

All five metrics are calculated during the same grid-search CV:
accuracy, precision, recall, F1 and ROC-AUC.

The best hyperparameters are selected using CV F1. The baseline and tuned candidates are compared using the same CV-based selection metric.

## Model Selection
The baseline and tuned candidates are compared using CV F1.
The final model is selected before the test set is evaluated.

## Bias-Variance Analysis
Training and cross-validation performance are compared.
A large train-CV gap is treated as a possible indicator of
higher variance/overfitting.

## Final Selected Model
{selected_model_name}

## Final Test Results
Accuracy: {test_accuracy:.4f}
Precision: {test_precision:.4f}
Recall: {test_recall:.4f}
F1: {test_f1:.4f}
ROC-AUC: {test_roc_auc:.4f}

## Limitation
The dataset is small and the final test set contains only
{len(y_test)} observations. Therefore, these results describe
performance on this experiment and should not be interpreted
as clinical validation or deployment-level evidence.
'''

strategy_path = os.path.join(
    BASE_DIR,
    "modeling_strategy.md"
)

with open(strategy_path, "w") as f:
    f.write(strategy_text)

print("Strategy report saved:", strategy_path)



Strategy report saved: /content/data_scientist_outputs/modeling_strategy.md


In [27]:
# ============================================================
# 24. FINAL EXPERIMENT SUMMARY
# ============================================================

summary = pd.DataFrame([
    {
        "Item": "Dataset",
        "Value": "UCI Heart Disease"
    },
    {
        "Item": "Total Samples",
        "Value": len(X)
    },
    {
        "Item": "Input Features",
        "Value": X.shape[1]
    },
    {
        "Item": "Training Samples",
        "Value": len(X_train)
    },
    {
        "Item": "Test Samples",
        "Value": len(X_test)
    },
    {
        "Item": "CV Strategy",
        "Value": "5-fold StratifiedKFold"
    },
    {
        "Item": "Selection Metric",
        "Value": "CV F1"
    },
    {
        "Item": "Selected Model",
        "Value": selected_model_name
    },
    {
        "Item": "Best CV F1",
        "Value": round(best_cv_f1, 4)
    },
    {
        "Item": "Test Accuracy",
        "Value": round(test_accuracy, 4)
    },
    {
        "Item": "Test Precision",
        "Value": round(test_precision, 4)
    },
    {
        "Item": "Test Recall",
        "Value": round(test_recall, 4)
    },
    {
        "Item": "Test F1",
        "Value": round(test_f1, 4)
    },
    {
        "Item": "Test ROC-AUC",
        "Value": round(test_roc_auc, 4)
    }
])

display(summary)

summary.to_csv(
    os.path.join(
        CSV_DIR,
        "experiment_summary.csv"
    ),
    index=False
)



,Item,Value
0,Dataset,UCI Heart Disease
1,Total Samples,303
2,Input Features,13
3,Training Samples,242
4,Test Samples,61
5,CV Strategy,5-fold StratifiedKFold
6,Selection Metric,CV F1
7,Selected Model,SVM
8,Best CV F1,0.827
9,Test Accuracy,0.8525


In [28]:
# ============================================================
# 25. DELIVERABLE CHECK
# ============================================================

print("=" * 70)
print("DATA SCIENTIST DELIVERABLE CHECK")
print("=" * 70)

print("✓ Modeling strategy")
print("✓ Baseline vs candidate comparison")
print("✓ 5-fold stratified cross-validation")
print("✓ Hyperparameter tuning")
print("✓ Bias-variance diagnostic")
print("✓ Final untouched test evaluation")
print("✓ Classification report")
print("✓ Confusion matrix")
print("✓ Serialized .pkl model")
print("✓ Model metadata")

print("\nGenerated CSV reports:")
for filename in sorted(os.listdir(CSV_DIR)):
    if filename.endswith(".csv"):
        print("  ✓", filename)

print("\nModel files:")
for filename in sorted(os.listdir(MODEL_DIR)):
    print("  ✓", filename)



DATA SCIENTIST DELIVERABLE CHECK
✓ Modeling strategy
✓ Baseline vs candidate comparison
✓ 5-fold stratified cross-validation
✓ Hyperparameter tuning
✓ Bias-variance diagnostic
✓ Final untouched test evaluation
✓ Classification report
✓ Confusion matrix
✓ Serialized .pkl model
✓ Model metadata

Generated CSV reports:
  ✓ algorithm_justification.csv
  ✓ baseline_cv_results.csv
  ✓ best_hyperparameters.csv
  ✓ bias_variance_diagnostic.csv
  ✓ classification_report.csv
  ✓ confusion_matrix.csv
  ✓ cross_validation_comparison.csv
  ✓ decision_tree_tuning_results.csv
  ✓ experiment_summary.csv
  ✓ final_model_comparison.csv
  ✓ final_test_metrics.csv
  ✓ model_selection_summary.csv
  ✓ random_forest_tuning_results.csv
  ✓ svm_tuning_results.csv
  ✓ target_distribution.csv
  ✓ test_predictions.csv
  ✓ train_test_split_summary.csv
  ✓ tuned_model_comparison.csv

Model files:
  ✓ heart_disease_final_model_v1.pkl
  ✓ model_metadata_v1.json


In [29]:
# ============================================================
# 26. CREATE DOWNLOAD PACKAGE
# ============================================================

DOWNLOAD_DIR = "/content/DS_repository_outputs"

if os.path.exists(DOWNLOAD_DIR):
    shutil.rmtree(DOWNLOAD_DIR)

os.makedirs(
    os.path.join(DOWNLOAD_DIR, "outputs", "data_scientist"),
    exist_ok=True
)

os.makedirs(
    os.path.join(DOWNLOAD_DIR, "models"),
    exist_ok=True
)

# Copy CSV reports
shutil.copytree(
    CSV_DIR,
    os.path.join(
        DOWNLOAD_DIR,
        "outputs",
        "data_scientist"
    ),
    dirs_exist_ok=True
)

# Copy modeling strategy
shutil.copy(
    strategy_path,
    os.path.join(
        DOWNLOAD_DIR,
        "outputs",
        "data_scientist",
        "modeling_strategy.md"
    )
)

# Copy model artifacts
shutil.copy(
    model_path,
    os.path.join(
        DOWNLOAD_DIR,
        "models",
        "heart_disease_final_model_v1.pkl"
    )
)

shutil.copy(
    metadata_path,
    os.path.join(
        DOWNLOAD_DIR,
        "models",
        "model_metadata_v1.json"
    )
)

zip_path = shutil.make_archive(
    "/content/DS_repository_outputs",
    "zip",
    DOWNLOAD_DIR
)

print("Package created:")
print(zip_path)



Package created:
/content/DS_repository_outputs.zip


In [30]:
# ============================================================
# 27. DOWNLOAD FROM GOOGLE COLAB
# ============================================================

from google.colab import files

files.download(
    "/content/DS_repository_outputs.zip"
)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>